In [ ]:
# Google Drive 마운트
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!unzip TL_81_단일.zip -d /content/TL_81_단일

In [ ]:
!unzip /content/new_images.zip -d /content/

In [20]:
import os, json, shutil
from PIL import Image

json_root = "/content/TL_81_단일/TL_81_단일"
img_root = "/content/new_images"
out_img_dir = "/content/new_yolo/images"
out_lbl_dir = "/content/new_yolo/labels"

os.makedirs(out_img_dir, exist_ok=True)
os.makedirs(out_lbl_dir, exist_ok=True)

success, fail = 0, []

for img_fname in os.listdir(img_root):
    stem = os.path.splitext(img_fname)[0].replace(' - 복사본', '').strip()
    kid = stem.split('_')[0]

    img_path = os.path.join(img_root, img_fname)
    json_path = os.path.join(json_root, kid + '_json', stem + '.json')

    try:
        with open(json_path, 'r', encoding='utf-8') as f:
            data = json.load(f)

        img_info = data['images'][0]
        W, H = img_info['width'], img_info['height']

        lines = []
        for ann in data['annotations']:
            x1, y1, w, h = ann['bbox']
            cx = (x1 + w / 2) / W
            cy = (y1 + h / 2) / H
            nw = w / W
            nh = h / H
            lines.append(f"0 {cx:.6f} {cy:.6f} {nw:.6f} {nh:.6f}")

        shutil.copy(img_path, os.path.join(out_img_dir, img_fname))
        with open(os.path.join(out_lbl_dir, stem + '.txt'), 'w') as f:
            f.write('\n'.join(lines))

        success += 1

    except Exception as e:
        fail.append((img_fname, str(e)))

print(f"✅ 변환 성공: {success}장")
print(f"❌ 실패: {len(fail)}장")
if fail:
    for fname, err in fail[:5]:
        print(f"  {fname}: {err}")

✅ 변환 성공: 500장
❌ 실패: 0장


In [29]:
import os, shutil

# dataset.yaml 생성
yaml_content = """path: /content/merged_yolo
train: train/images
val: val/images

nc: 1
names:
  0: pill
"""

yaml_path = "/content/merged_yolo/dataset.yaml"
with open(yaml_path, 'w') as f:
    f.write(yaml_content)

print("✅ dataset.yaml 생성 완료")
print(yaml_content)

# Google Drive에 저장
dst = "/content/drive/MyDrive/data/초급_프로젝트/yolo_dataset_v2"
if os.path.exists(dst):
    shutil.rmtree(dst)
shutil.copytree("/content/merged_yolo", dst)

print(f"✅ Google Drive 저장 완료: {dst}")

# 최종 확인
for split in ['train', 'val']:
    for sub in ['images', 'labels']:
        n = len(os.listdir(f"{dst}/{split}/{sub}"))
        print(f"  {split}/{sub}: {n}개")

✅ dataset.yaml 생성 완료
path: /content/merged_yolo
train: train/images
val: val/images

nc: 1
names:
  0: pill

✅ Google Drive 저장 완료: /content/drive/MyDrive/data/초급_프로젝트/yolo_dataset_v2
  train/images: 1740개
  train/labels: 1740개
  val/images: 231개
  val/labels: 231개
